In [1]:
import csv
import re
import numpy as np
import pickle

In [ ]:
def read_articles(file_path):
    articles = []
    with open(file_path, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            articles.append(row)
    return articles


raw_articles = read_articles('articles.csv')
print(f"{len(raw_articles)} articles is downloaded")

20 articles is downloaded


In [11]:
def clean_content(text):
    text = text.lower()
    text = re.sub(r'[\d\W_]+', ' ', text)
    words = text.split()
    return words


example_words = clean_content(raw_articles[0]['content'])
print(f"sample of cleaned wordes :{example_words[:5]}")

sample of cleaned wordes :['artificial', 'intelligence', 'is', 'transforming', 'industries']


In [12]:
def build_global_vocabulary(articles):
    vocab = set()
    for art in articles:
        words = clean_content(art['content'])
        vocab.update(words)
    return sorted(list(vocab))

global_vocab = build_global_vocabulary(raw_articles)
print(f"number of uniqe wordes: {len(global_vocab)}")

number of uniqe wordes: 167


In [13]:
def get_vector_representation(content_words, vocab):
   
    vector = [1 if word in content_words else 0 for word in vocab]
    return np.array(vector)


article_vectors = [get_vector_representation(clean_content(a['content']), global_vocab) for a in raw_articles]
print(f"shape of the first vector:{article_vectors[0].shape}")

shape of the first vector:(167,)


In [ ]:
def calculate_similarity_matrix(vectors):
    n = len(vectors)
    matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            A = vectors[i]
            B = vectors[j]
            
            
            dot_product = np.dot(A, B)
            norm_a = np.linalg.norm(A)
            norm_b = np.linalg.norm(B)
            
            if norm_a > 0 and norm_b > 0:
                matrix[i][j] = dot_product / (norm_a * norm_b)
            else:
                matrix[i][j] = 0.0
    return matrix

sim_matrix = calculate_similarity_matrix(article_vectors)

In [14]:

with open('similarities.pkl', 'wb') as f:
    pickle.dump(sim_matrix, f)


def find_most_similar(article_id, articles, matrix, top_n=3):
    idx = -1
    for i, art in enumerate(articles):
        if int(art['id']) == article_id:
            idx = i
            break
            
    if idx == -1: return "Article ID not found"

    similarities = []
    for i in range(len(matrix)):
        if i != idx:
            similarities.append((i, matrix[idx][i]))
    
   
    similarities.sort(key=lambda x: x[1], reverse=True)
    
   
    top_titles = [articles[item[0]]['title'] for item in similarities[:top_n]]
    return top_titles


recommendations = find_most_similar(2, raw_articles, sim_matrix)
print(f"most similar articles'{raw_articles[0]['title']}':")
for title in recommendations:
    print(f"- {title}")

most similar articles'The Rise of AI':
- Industrial Robotics
- The Rise of AI
- Reinforcement Learning
